# Projeto 1 — Data Lake E-commerce Analytics

## 1. Introdução

Este projeto tem como objetivo desenvolver uma solução de Engenharia de Dados voltada à organização, ao processamento e à análise de informações de vendas de um e-commerce.

A partir de uma base de dados em formato CSV, foram realizados processos de extração, limpeza, validação e transformação utilizando Python e suas bibliotecas de manipulação de dados.

A solução utiliza uma arquitetura de Data Lake na AWS, organizada em três camadas: Bronze, Silver e Gold. Os dados são armazenados no Amazon S3, catalogados pelo AWS Glue e consultados por meio do Amazon Athena utilizando SQL.

## 2. Objetivos

- Explorar uma base de dados de vendas.
- Identificar inconsistências e valores ausentes.
- Realizar limpeza e transformação dos dados.
- Aplicar validações e remover duplicidades.
- Converter os dados tratados para o formato Parquet.
- Implementar as camadas Bronze, Silver e Gold.
- Realizar consultas analíticas com SQL.
- Disponibilizar indicadores para visualização no Power BI.

## 3. Tecnologias utilizadas

- Python e Pandas.
- NumPy.
- PyArrow.
- Google Colab.
- Amazon S3.
- AWS Glue.
- Amazon Athena.
- Power BI.

## 4. Arquitetura do projeto

CSV → S3 Bronze → Python/Pandas → S3 Silver (Parquet) → AWS Glue → Amazon Athena → S3 Gold → Power BI.

## 5. Resultado esperado

Construir um Data Lake capaz de organizar dados de vendas em diferentes níveis de processamento, disponibilizando informações estruturadas para consultas SQL e análises de desempenho comercial.

In [ ]:
!pip install pyarrow

In [ ]:
import pandas as pd
import numpy as np

print("Ambiente preparado!")

Ambiente preparado!


In [ ]:
df = pd.read_csv("dataset_ecommerce_bronze.csv")

print("Quantidade de registros:", len(df))
print("Quantidade de colunas:", len(df.columns))

df.head()

Quantidade de registros: 1000
Quantidade de colunas: 11


,id_pedido,data_pedido,cliente,produto,categoria,quantidade,preco_unitario,desconto,valor_total,regiao,status
0,PED00001,2025-03-13,Cliente 053,Smartphone,Eletrônicos,1,2101.01,0.00,2101.01,Sul,Concluído
1,PED00002,2025-09-16,Cliente 014,Cadeira,Móveis,1,689.05,0.00,689.05,Sul,Concluído
2,PED00003,2025-01-04,Cliente 217,Mochila,Acessórios,1,158.37,0.00,158.37,Centro-Oeste,Concluído
3,PED00004,2025-07-14,Cliente 184,Monitor,Eletrônicos,4,918.95,0.00,3675.80,Centro-Oeste,Enviado
4,PED00005,2025-02-10,Cliente 151,Notebook,Eletrônicos,2,3525.36,0.05,6698.18,Sul,Em processamento


Analise da Qtd de dados

In [ ]:
print("Valores nulos:")
print(df.isnull().sum())

print("\nDuplicados:")
print(df.duplicated().sum())

print("\nTipos de dados:")
print(df.dtypes)

Valores nulos:
id_pedido         0
data_pedido       0
cliente           0
produto           0
categoria         0
quantidade        0
preco_unitario    0
desconto          0
valor_total       0
regiao            0
status            0
dtype: int64

Duplicados:
0

Tipos de dados:
id_pedido          object
data_pedido        object
cliente            object
produto            object
categoria          object
quantidade          int64
preco_unitario    float64
desconto          float64
valor_total       float64
regiao             object
status             object
dtype: object


Tranformando os dados

In [ ]:
# Converter data para datetime
df["data_pedido"] = pd.to_datetime(
    df["data_pedido"],
    errors="coerce"
)

# Converter quantidade para inteiro
df["quantidade"] = df["quantidade"].astype("int64")

# Converter valores monetários
colunas_valores = [
    "preco_unitario",
    "desconto",
    "valor_total"
]

for coluna in colunas_valores:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

# Recalcular valor total
df["valor_total"] = (
    df["quantidade"] *
    df["preco_unitario"] *
    (1 - df["desconto"])
).round(2)

# Remover duplicados pelo ID do pedido
df = df.drop_duplicates(subset=["id_pedido"])

print("Dados tratados:", len(df))

Dados tratados: 1000


Converter para Parquet

In [ ]:
df.to_parquet(
    "pedidos_silver.parquet",
    index=False
)

print("Arquivo Parquet gerado!")

Arquivo Parquet gerado!


Validar o arquivo

In [ ]:
df_silver = pd.read_parquet("pedidos_silver.parquet")

df_silver.head()

,id_pedido,data_pedido,cliente,produto,categoria,quantidade,preco_unitario,desconto,valor_total,regiao,status
0,PED00001,2025-03-13,Cliente 053,Smartphone,Eletrônicos,1,2101.01,0.00,2101.01,Sul,Concluído
1,PED00002,2025-09-16,Cliente 014,Cadeira,Móveis,1,689.05,0.00,689.05,Sul,Concluído
2,PED00003,2025-01-04,Cliente 217,Mochila,Acessórios,1,158.37,0.00,158.37,Centro-Oeste,Concluído
3,PED00004,2025-07-14,Cliente 184,Monitor,Eletrônicos,4,918.95,0.00,3675.80,Centro-Oeste,Enviado
4,PED00005,2025-02-10,Cliente 151,Notebook,Eletrônicos,2,3525.36,0.05,6698.18,Sul,Em processamento
